In [1]:
import os
import sys
import pandas as pd
import numpy as np
import time

# Ensure the workspace root is loaded into the interpreter path mapping
local_path = os.path.dirname(os.path.abspath('__file__'))
if local_path not in sys.path:
    sys.path.insert(0, local_path)

# Import your decoupled system components exactly as named
from TES_Fixed_Building_Orchestrator import run_building_simulation as run_fixed_orchestrator
from TES_Dynamic_Building_Orchestrator import run_dynamic_building_simulation as run_dynamic_orchestrator
from TES_Simulation_Engine_FIXED import run_tes_sizing_engine

def execute_master_analysis_pipeline():
    print("=====================================================================")
    print("   STAGE 1: EVALUATING SCENARIO A - FIXED GRADIENT OPTIMIZATION      ")
    print("=====================================================================")
    
    best_hot_volume = float('inf')
    best_hour_block = None
    winning_metrics = None
    
    start_time = time.perf_counter()
    
    # Sweep across all 24 hours of the day to test consecutive 6-hour heating/cooling blocks
    for start_hour in range(24):
        # 1. Build candidate daily floor profile array
        candidate_day = [0.0] * 24
        for hr in range(6):
            candidate_day[(start_hour + hr) % 24] = 1.0
        
        # Extend day loop across 7 days to form a full weekly baseline profile (length 168)
        candidate_weekly_profile = candidate_day * 7
        
        # 2. Run the multi-zone building simulation loop using this schedule block
        run_fixed_orchestrator(
            custom_floor_profile=candidate_weekly_profile,
            silent_mode=False,
            export_excel=False
        )
        
        # 3. Call the bisection search function to size the thermal tank for this demand profile
        metrics = run_tes_sizing_engine(
            thermal_data_file='Output - Results_Yearly_Zone_Demands_Combined - FIXED.csv',
            T_START_LIST=[21.0, 22.0, 21.0, 21.0, 21.0],
            T_SET_LIST=[21.0, 22.0, 21.0, 21.0, 21.0],
            T_SWING_LIST=[2.0, 0.0, 2.0, 2.0, 2.0],
            SILENT_MODE=True
        )
        
        current_hot_volume = metrics["Hot_Tank_Volume_Vc_m3"]
        current_cold_volume = metrics["Cold_Tank_Volume_Vc_m3"]
        
        # LIVE CONSOLE FEEDBACK PROGRESS MAP (Tracks both Heating and Cooling Footprints)
        print(f"[{start_hour+1:02d}/24] Evaluated UFH Block Starting at {start_hour:02d}:00 -> Required Hot Tank: {current_hot_volume:.2f} m³ | Cold Tank: {current_cold_volume:.2f} m³")
        
        # 4. Minimize physical volume tracking
        if current_hot_volume < best_hot_volume:
            best_hot_volume = current_hot_volume
            best_hour_block = start_hour
            winning_metrics = metrics

    fixed_end_time = time.perf_counter()
    
    # =====================================================================
    # DISPLAY WINNING FIXED SIMULATION DATA METRICS
    # =====================================================================
    print("\n" + "="*20 + " FIXED OPTIMIZATION SUCCESS " + "="*20)
    print(f"Total Sweep Execution Time: {fixed_end_time - start_time:.2f} seconds")
    print(f"Winning Underfloor Heating/Cooling Start Time Block: {best_hour_block:02d}:00")
    print(f"Absolute Minimum Hot Tank Sizing Capacity ($E_{{\\text{{max\\_gap}}}}$): {winning_metrics['Hot_Tank_Capacity_kWh']:.2f} kWh")
    print(f"Physical Hot Footprint Volume ($V_c$): {winning_metrics['Hot_Tank_Volume_Vc_m3']:.2f} m³")
    print(f"Hydraulic Balancing Hot Buffer Volume ($V_h$): {winning_metrics['Hot_Tank_Volume_Vh_m3']:.2f} m³")
    print("=" * 62 + "\n")

    # =====================================================================
    # GENERATE DETAILED DICTATING WINDOW TABLES FOR THE CHAMPION FIXED RUN
    # =====================================================================
    print("Generating comprehensive final reports and dictating window tables for Champion Fixed configuration...")
    champion_fixed_profile = [0.0] * 24
    for hr in range(6):
        champion_fixed_profile[(best_hour_block + hr) % 24] = 1.0
        
    run_fixed_orchestrator(custom_floor_profile=champion_fixed_profile * 7, silent_mode=False, export_excel=False)
    
    # Executing with SILENT_MODE=False forces the engine to push Table 1 & Table 2 layouts to the screen
    run_tes_sizing_engine(
        thermal_data_file='Output - Results_Yearly_Zone_Demands_Combined - FIXED.csv',
        T_START_LIST=[21.0, 22.0, 21.0, 21.0, 21.0],
        T_SET_LIST=[21.0, 22.0, 21.0, 21.0, 21.0],
        T_SWING_LIST=[2.0, 0.0, 2.0, 2.0, 2.0],
        SILENT_MODE=False
    )

if __name__ == "__main__":
    execute_master_analysis_pipeline()

   STAGE 1: EVALUATING SCENARIO A - FIXED GRADIENT OPTIMIZATION      
Simulation Completed successfully. Energy tables and Dynamic Grid Profiles exported.
[01/24] Evaluated UFH Block Starting at 00:00 -> Required Hot Tank: 2.34 m³ | Cold Tank: 2.57 m³
Simulation Completed successfully. Energy tables and Dynamic Grid Profiles exported.
[02/24] Evaluated UFH Block Starting at 01:00 -> Required Hot Tank: 2.34 m³ | Cold Tank: 1.76 m³
Simulation Completed successfully. Energy tables and Dynamic Grid Profiles exported.
[03/24] Evaluated UFH Block Starting at 02:00 -> Required Hot Tank: 2.34 m³ | Cold Tank: 1.76 m³
Simulation Completed successfully. Energy tables and Dynamic Grid Profiles exported.
[04/24] Evaluated UFH Block Starting at 03:00 -> Required Hot Tank: 2.34 m³ | Cold Tank: 1.76 m³
Simulation Completed successfully. Energy tables and Dynamic Grid Profiles exported.
[05/24] Evaluated UFH Block Starting at 04:00 -> Required Hot Tank: 2.34 m³ | Cold Tank: 1.76 m³
Simulation Completed

Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
331,105.30,0,112.16,112.16,17.24,0,1,0.00,19.50,21,0,1.68,34.70,22,0,2.99,15.80,21,0,1.36,20.60,21,0,1.77,14.70,21,0,1.27,STRUCT,0,0
332,85.40,0,66.94,66.94,9.78,0,0,0,11.40,20.96,0,0.77,34.50,22,0,2.97,4.30,20.99,0,0.29,20.60,20.95,0,1.39,14.60,20.95,0,0.99,Discharge,18.46,1.59
333,83.90,0,67.41,67.41,3.16,0,0,0,11.20,20.92,0,0.78,33.70,22,0,2.90,4.10,20.98,0,0.28,20.40,20.91,0,1.41,14.50,20.90,0,1.00,Discharge,16.49,1.42
334,76.60,0,72.58,72.58,1.76,0,0,0,11.10,20.91,0,0.91,26.70,22,0,2.30,4.10,20.98,0,0.33,20.30,20.90,0,1.66,14.40,20.89,0,1.18,Discharge,4.02,0.35
335,77.10,0,72.45,72.45,0.13,0,0,0,11.20,20.90,0,0.91,26.90,22,0,2.32,4.10,20.97,0,0.33,20.40,20.88,0,1.65,14.50,20.87,0,1.17,Discharge,4.65,0.40
336,102.90,0,125,125,0.13,0,620,0.77,19.10,20.98,1.44,2.07,26.20,22,0,2.26,23.10,21.00,1.44,2.15,20.20,20.98,1.44,2.51,14.30,20.98,1.44,1.78,STRUCT,0,0
337,118.10,0,125,125,2.62,0,619,2.48,24.50,21,1.20,2.20,36.10,22,0,3.11,23.10,21,1.20,2.02,20.10,21,1.20,1.89,14.30,21,1.20,1.34,STRUCT,0,0
338,117.50,0,125,125,10.12,0,618,1.95,24.40,21,0,2.10,35.90,22,0,3.09,23,21,0,1.98,20,21,0,1.72,14.20,21,0,1.22,STRUCT,0,0
339,118.10,0,125,125,17.02,0,617,1.03,24.50,21,0,2.11,36.10,22,0,3.11,23.10,21,0,1.99,20.10,21,0,1.73,14.30,21,0,1.23,STRUCT,0,0
340,118.90,0,125,125,17.24,0,616,0.04,24.70,21,0,2.13,36.40,22,0,3.13,23.20,21,0,2.00,20.20,21,0,1.74,14.40,21,0,1.24,STRUCT,0,0



============================== TABLE 2: HYDRAULIC STABILITY (Vh_H) ==============================


Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
330,104.20,0,112.74,112.74,17.24,0,2,0.00,19.30,21,0,1.66,34.10,22,0,2.94,15.70,21,0,1.35,20.50,21,0,1.77,14.60,21,0,1.26,STRUCT,0,0
331,105.30,0,112.16,112.16,17.24,0,1,0.00,19.50,21,0,1.68,34.70,22,0,2.99,15.80,21,0,1.36,20.60,21,0,1.77,14.70,21,0,1.27,STRUCT,0,0
332,85.40,0,66.94,66.94,9.78,0,0,0,11.40,20.96,0,0.77,34.50,22,0,2.97,4.30,20.99,0,0.29,20.60,20.95,0,1.39,14.60,20.95,0,0.99,Discharge,18.46,1.59
333,83.90,0,67.41,67.41,3.16,0,0,0,11.20,20.92,0,0.78,33.70,22,0,2.90,4.10,20.98,0,0.28,20.40,20.91,0,1.41,14.50,20.90,0,1.00,Discharge,16.49,1.42
334,76.60,0,72.58,72.58,1.76,0,0,0,11.10,20.91,0,0.91,26.70,22,0,2.30,4.10,20.98,0,0.33,20.30,20.90,0,1.66,14.40,20.89,0,1.18,Discharge,4.02,0.35


COOLING SIDE:
Optimized Tank (Vc): 15.11 kWh
Charging: Proportional | Discharge: Proportional
Final Volume (Vc): 2.57 m³
Final Volume (Vh): 2.52 m³
---------------------------------
ZONE DISCHARGE STRATEGIES (C):
Zone 1: ALLOW DRIFT from 21.0°C up to 23.00°C
Zone 2: ALLOW DRIFT from 22.0°C up to 22.00°C
Zone 3: ALLOW DRIFT from 21.0°C up to 23.00°C
Zone 4: ALLOW DRIFT from 21.0°C up to 23.00°C
Zone 5: ALLOW DRIFT from 21.0°C up to 23.00°C
---------------------------------

============================== TABLE 1: ENERGY CAPACITY (Vc_C) ==============================


Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
4092,0,63.40,65,65,15.11,0,1,0.00,12.40,21,0,1.33,0,22,0,0,9.90,21,0,1.07,24.60,21,0,2.65,16.50,21,0,1.78,STRUCT,0,0
4093,0,65.90,65,65,15.10,0,0,0,13.10,21.00,0,1.39,0,22,0,0,22.70,21.00,0,2.41,18,21.00,0,1.91,12.10,21.00,0,1.28,Discharge,0.90,0.10
4094,0,80.10,65,65,15.10,0,0,0,14,21.05,0,1.22,0,22,0,0,23.30,21.05,0,2.04,25.70,21.06,0,2.25,17.10,21.06,0,1.49,Discharge,15.10,1.63
4095,0,82.40,65,65,15.10,0,0,0,14.40,21.10,0,1.22,0,22,0,0,24,21.11,0,2.04,26.70,21.12,0,2.27,17.30,21.12,0,1.47,Discharge,17.40,1.87
4096,0,101.40,65,65,15.10,0,0,0,31.20,21.30,0,2.15,0,22,0,0,24.80,21.22,0,1.71,27.90,21.23,0,1.93,17.50,21.22,0,1.21,Discharge,36.40,3.92
4097,0,102.30,65,65,15.10,0,0,0,32,21.51,0,2.19,0,22,0,0,25.40,21.33,0,1.74,27.60,21.34,0,1.89,17.30,21.33,0,1.18,Discharge,37.30,4.02
4098,0,74.40,65,65,15.10,0,0,0,31.90,21.58,0,3.00,0,22,0,0,25.20,21.36,0,2.37,11,21.35,0,1.03,6.30,21.34,0,0.59,Discharge,9.40,1.01
4099,0,76.70,65,65,15.10,0,0,0,32.80,21.66,0,2.99,0,22,0,0,25.50,21.41,0,2.33,12.20,21.37,0,1.11,6.20,21.36,0,0.57,Discharge,11.70,1.26
4100,0,108.80,65,65,15.09,0,0,0,56.60,22.06,0,3.64,0,22,0,0,43.50,21.61,0,2.80,5.80,21.40,0,0.37,2.90,21.38,0,0.19,Discharge,43.80,4.72
4101,0,100.80,65,65,15.09,0,0,0,54,22.40,0,3.75,0,22,0,0,41.50,21.79,0,2.88,3.80,21.41,0,0.26,1.50,21.38,0,0.10,Discharge,35.80,3.85



============================== TABLE 2: HYDRAULIC STABILITY (Vh_C) ==============================


Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
4173,0,93.50,65,65,15.10,0,0,0,49.60,21.61,0,3.71,0,22,0,0,39,21.32,0,2.92,3.40,21.04,0,0.25,1.50,21.03,0,0.11,Discharge,28.50,3.07
4174,0,91.60,65,65,15.10,0,0,0,48.80,21.86,0,3.73,0,22,0,0,38.40,21.45,0,2.93,3.10,21.05,0,0.24,1.30,21.04,0,0.10,Discharge,26.60,2.86
4175,0,139.80,65,65,15.10,0,0,0,63.40,22.46,0,3.17,0,22,0,0,54.40,21.79,0,2.72,11.50,21.12,0,0.58,10.50,21.13,0,0.53,Discharge,74.80,8.05
4176,0.10,67.50,65,65,15.10,0,0,0,36.90,22.48,0,3.83,0,22,0,0,20,21.80,0,2.07,10.60,21.12,0,1.10,0,21.13,0,0,Discharge,2.50,0.27
4177,0.30,56.60,65,65,15.11,0,15,0.00,19.10,22.41,12.05,2.50,9.30,22,0,1.00,18.50,21.76,9.79,2.35,9.70,21.11,1.57,1.10,0,21.12,1.13,0.04,TES,0,0


Performance: 8031.62 ms (1 runs total)
